#### RFE and Mutual Information to choose optimal variable for the dataset (and study potentially redundant ones)

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, mutual_info_regression, mutual_info_classif
from sklearn.model_selection import train_test_split
import seaborn as sns

# load data
df = pd.read_csv("../heart.csv")

# some missing values, let's drop them for now
df = df.dropna()

# change this to your own target variable
TARGET_VARIABLE = "Rating"

#### RFE - recursive feature elimination

In [ ]:
# typical X/y -split
X = df.drop(TARGET_VARIABLE, axis=1)
y = df[TARGET_VARIABLE]

# define model (linear regression, random forest, XGBoost etc.)
# technically you can use pretty much any classic ML algorithm
# model = LinearRegression()

# idea: you might want to try RFE with multuple ML algorithms
# and then cross-validate the results (which results are common in all models)

# another idea: if you plan on using e.g. XGBoost in your final model
# it's probably a good idea to use the same XGBoost here
model = RandomForestRegressor()

# create RFE, place the model and choose number of optimal variables
rfe = RFE(estimator=model, n_features_to_select=7)

# fit the RFE model with our data
rfe.fit(X, y)

# get rankings and results
rankings = rfe.ranking_
support = rfe.support_

# build a DataFrame to wrap up result for easier inspection
results_df = pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rankings,
    "Selected": support
}).sort_values(by="Ranking")

# you can use these results with any other knowledge you have
# from other optimal variable selection tools, and cross-validation
results_df

In [ ]:

# the selection loosely follow the correlations it seems
# this might imply the data is more or less linear or something else
df.corr()[TARGET_VARIABLE].sort_values(ascending=False)

#### Mutual information - another alternative

In [ ]:
# fit the mutual information algorithm
# => that's why either or works
mi = mutual_info_regression(X, y)

# convert results into DataFrame
mi_results = pd.Series(mi, index=X.columns).sort_values(ascending=False)

# high value => variable is strongly connected to target variable
# low value => weak connection
# e.g. very similar to phik-matrix
mi_results